In [1]:
# ==========================================
# MASTER ASSEMBLY SCRIPT (FOR ML HANDOFF)
# ==========================================
import pandas as pd
import os

print("Assembling the Final Master Dataset for Member 4...")
data_path = "../data"

# 1. Load the three audited CSVs
rf_df = pd.read_csv(os.path.join(data_path, "customer_rf_metrics.csv"))
monetary_df = pd.read_csv(os.path.join(data_path, "member2_monetary.csv"))
features_df = pd.read_csv(os.path.join(data_path, "member3_features.csv"))

# 2. Strict ID Type Casting (The final armor)
rf_df['customer_unique_id'] = rf_df['customer_unique_id'].astype(str)
monetary_df['customer_unique_id'] = monetary_df['customer_unique_id'].astype(str)
features_df['customer_unique_id'] = features_df['customer_unique_id'].astype(str)

# 3. The Grand Merge (Outer joins to prevent losing ANY customer data)
master_df = rf_df.merge(monetary_df, on="customer_unique_id", how="outer")
master_df = master_df.merge(features_df, on="customer_unique_id", how="outer")

# 4. Final Null Annihilation (Outer joins create NaNs for mismatched rows)
# We fill them safely so the ML model receives flawless math
master_df["Recency"] = master_df["Recency"].fillna(master_df["Recency"].max()) # Missing recency = worst possible recency
master_df["Frequency"] = master_df["Frequency"].fillna(0)
master_df["Monetary"] = master_df["Monetary"].fillna(0.0)
master_df["Top_Category"] = master_df["Top_Category"].fillna("unknown_category")
master_df["avg_review_score"] = master_df["avg_review_score"].fillna(3.0)
master_df["review_count"] = master_df["review_count"].fillna(0)
master_df["low_review_flag"] = master_df["low_review_flag"].fillna(0).astype(int)

# 5. Hand-off Export
output_file = os.path.join(data_path, "final_master_dataset.csv")
master_df.to_csv(output_file, index=False)

print(f"Success! The Master Dataset is perfectly assembled: {master_df.shape}")
print(f"Handoff file saved to: {output_file}")
display(master_df.head(3))

Assembling the Final Master Dataset for Member 4...
Success! The Master Dataset is perfectly assembled: (93358, 8)
Handoff file saved to: ../data\final_master_dataset.csv


,customer_unique_id,Recency,Frequency,Monetary,Top_Category,avg_review_score,review_count,low_review_flag
0,0000366f3b9a7992bf8c76cfdf3221e2,112,1,141.90,cama_mesa_banho,5.0,1,0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,115,1,27.19,beleza_saude,4.0,1,0
2,0000f46a3911fa3c0805444483337064,537,1,86.22,papelaria,3.0,1,0
